In [9]:
import mne
import numpy as np
import addcopyfighandler
import matplotlib.pyplot as plt
#기초적인 라이브러리

In [10]:
%matplotlib qt

In [11]:
from pathlib import Path
#Path 지정을 위한 라이브러리

data_path = Path("//HCN-NAS/ywYang/s02")

raw_file = data_path / "hri-s2-b1-[2026.03.30-11.11.42].gdf"

raw = mne.io.read_raw_gdf(raw_file, preload=True)
#gdf 파일이므로 read_raw_edf 가 아닌 read_raw_gdf 사용
montage = mne.channels.make_standard_montage("standard_1020")
#표준 전극 배치도(10-20 시스템)를 불러와서 montage라는 변수에 저장
raw.set_montage(montage, on_missing='ignore')
#불러온 전극 위치 지도를 실제 데이터(raw)의 각 채널 이름과 연결
#EXG1~9 on_missing ignore


Extracting GDF parameters from \\HCN-NAS\ywYang\s02\hri-s2-b1-[2026.03.30-11.11.42].gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fp2, F4, Fz, F3, T7, C3, Cz, C4, T8, P4, Pz, P3, O1, Oz, O2, EXG1, EXG2, EXG3, EXG4, EXG5, EXG6, EXG7, EXG8
Creating raw.info structure...
Reading 0 ... 1041535  =      0.000 ...  1017.124 secs...


<RawGDF | hri-s2-b1-[2026.03.30-11.11.42].gdf, 25 x 1041536 (1017.1 s), ~198.7 MiB, data loaded>

In [12]:
raw_eeg = raw.copy().drop_channels(
    ['EXG1', 'EXG2', 'EXG3', 'EXG4',
     'EXG5', 'EXG6', 'EXG7', 'EXG8']
)
#EXG에는 주로 EOG, EMG, ECG등이 있으므로 drop하기 

In [13]:
print(raw)
#raw를 출력하라
print(raw.info)
#raw의 info(정보)를 출력하라

<RawGDF | hri-s2-b1-[2026.03.30-11.11.42].gdf, 25 x 1041536 (1017.1 s), ~198.7 MiB, data loaded>
<Info | 9 non-empty values
 bads: []
 ch_names: Fp1, Fp2, F4, Fz, F3, T7, C3, Cz, C4, T8, P4, Pz, P3, O1, Oz, ...
 chs: 24 EEG, 1 Stimulus
 custom_ref_applied: False
 dig: 19 items (3 Cardinal, 16 EEG)
 highpass: 0.0 Hz
 lowpass: 512.0 Hz
 meas_date: unspecified
 nchan: 25
 projs: []
 sfreq: 1024.0 Hz
 subject_info: <subject_info | his_id: X, last_name: >
>


In [17]:
raw.compute_psd(fmax = 512).plot(picks="data", exclude= "bads", amplitude=False)
#raw의 power spectral density를 522Hz 이하만 잡기 - lowpass filter로 512Hz 해주었기 때문에
#data type(eeg)을 선택, bad channel을 배제(없다), power로 표시해서 그래프로 그려라
raw.plot(duration=5, n_channels=25)
#한 화면에 5초 길이 25개의 channel을 그래프로 그려라

Effective window size : 2.000 (s)
Plotting power spectral density (dB=True).


c:\Users\user\anaconda3\envs\mne\Lib\site-packages\mne_qt_browser\_pg_figure.py:1544: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z = zscore(data, axis=1)


<mne_qt_browser._pg_figure.MNEQtBrowser(0x1b9105b04b0) at 0x000001B926C26F00>

c:\Users\user\anaconda3\envs\mne\Lib\site-packages\mne_qt_browser\_pg_figure.py:1544: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z = zscore(data, axis=1)


In [16]:

#set up and it the ICA
ica = mne.preprocessing.ICA(n_components=10, random_state=40, max_iter=800)
#ica components의 개수를 10개로 설정, 
#난수를 40로 설정하여 다음 차례에도 비슷한 값이 나오도록 설정
#ICA 알고리즘이 수렴할 때까지 최대 800번 허용

ica.fit(raw_eeg)
ica.exclude = [0,1,2,3,4,5,6,7,8,9] #details on how we picked these are omitted here
ica.plot_properties(raw, picks=ica.exclude)

Fitting ICA to data using 16 channels (please be patient, this may take a while)


C:\Users\user\AppData\Local\Temp\ipykernel_14120\2331615514.py:7: RuntimeWarning: The data has not been high-pass filtered. For good ICA performance, it should be high-pass filtered (e.g., with a 1.0 Hz lower bound) before fitting ICA.
  ica.fit(raw_eeg)


Selecting by number: 10 components
Fitting ICA took 142.0s.


c:\Users\user\anaconda3\envs\mne\Lib\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
508 matching events found
No baseline correction applied
0 pro

[<Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>,
 <Figure size 1400x1200 with 6 Axes>]